# Feature selection of train data

In [ ]:
import numpy as np
import pandas as pd
import toad
import matplotlib.pyplot as plt
from aidataprocess import AiDataProcess
from aifeatureminer import AiFeatureMiner
import hashlib

data=pd.read_csv("")
myad=AiDataProcess(data)
drop_list=[]
my_quality=myad.my_simple_select(to_drop=drop_list,target='label',only_ive=True)
print(my_quality)
data2=myad.get_dataframe()
data2=data2.reset_index(drop=False)
# 1. 实例化特征挖掘/处理的类，传入经过前期清洗和预处理的数据集 kaifa2
myfea = AiFeatureMiner(data2)

# 2. 定义一个不需要参与特征筛选的列名列表，通常是主键ID、索引列或日期列
drop_list = ['index']

# 3. 调用核心特征筛选方法，这一步会自动按照缺失率、IV值、相关性等指标进行降维和特征淘汰
myfea.my_feature_select(
    to_drop=drop_list,  # 排除列：指定 drop_list 中的列不参与特征筛选（直接保留或忽略）
    y_target='label',   # 目标变量：指定数据集中的预测标签（如好坏样本0/1），用于后续计算IV值和相关性
    empty_p=0.99,       # 缺失率过滤（Empty Percentage）：剔除缺失率大于 99% 的特征（包含有效信息太少）
    iv_p=0.02,          # IV值过滤（Information Value）：剔除 IV值小于 0.02 的特征（即对目标变量缺乏预测区分度的无用特征）
    corr_p=0.8,         # 相关性过滤（Correlation）：如果两个特征之间的相关系数大于 0.8，说明它们提供的信息高度重合/冗余，算法会自动剔除其中IV值较低的一个
    must_var=[]         # 强制保留特征（白名单）：写在这里的列名，即使触碰了上述缺失率、IV值或相关性的淘汰规则，也会被强行保留
)

# 4. 获取特征筛选完毕后的干净数据集
selected_data = myfea.selected_dataframe

# 5. （可选）根据业务需求，对筛选后的数据集进行最后的手动微调，手动剔除某些不需要的列
# axis=1 表示操作的是“列”而不是“行”
selected_data = selected_data.drop(['需要额外删除的列名'], axis=1)
# 将筛选后的数据保存为 CSV 文件
# index=False 的作用是：不要把 DataFrame 最左侧那一列无意义的行号（0, 1, 2...）当作新的一列保存进去
selected_data.to_csv('feature_selected_data.csv', index=False)

print("✅ 数据已成功保存为 feature_selected_data.csv")

# Test data feature align 2 train data feature

In [ ]:
train_selected_data = myfea.selected_dataframe

# 获取训练集最终决定保留下来的“特征白名单”
# 注意：要把目标变量 'label' 也包含进去，方便后续评估模型
final_features_list = train_selected_data.columns.tolist()

print(f"训练集筛选完毕，共保留 {len(final_features_list)} 个字段")

# ==========================================
# 阶段二：在【测试集】上直接套用规则（切忌重新跑筛选逻辑）
# ==========================================
# 1. 读取测试集原始数据（或者经过了和训练集一模一样的基础清洗后的数据）
test_data = pd.read_csv("你的测试集.csv")

# 2. 核心操作：直接用训练集的“白名单”去截取测试集
# 这样不仅保证了特征 100% 一致，还避免了数据泄露
test_selected_data = test_data[final_features_list]

# 3. 分别保存，准备喂给模型
train_selected_data.to_csv('train_selected.csv', index=False)
test_selected_data.to_csv('test_selected.csv', index=False)

print("✅ 训练集和测试集特征已完全对齐并保存！")

# Feature Selection Optuna

In [2]:
#Lightgbm
import optuna
from sklearn.model_selection import cross_val_score
from lightgbm import LGBMClassifier 
import pandas as pd
import numpy as np
import pandas as pd
import toad
import matplotlib.pyplot as plt
from aidataprocess import AiDataProcess
from aifeatureminer import AiFeatureMiner
import hashlib

data = pd.read_csv('F:\\datamining\\sensor_data260624\\normalize_feature\\feature\\combined_features.csv')
print(f"数据集加载完成，数据形状: {data.shape}")
def objective(trial):
    # 1. 让 Optuna 智能“建议”参数，而不是死板的列表
    empty_p = trial.suggest_float('empty_p', 0.90, 0.99)
    iv_p = trial.suggest_float('iv_p', 0.01, 0.05)
    corr_p = trial.suggest_float('corr_p', 0.70, 0.95)
    
    # 2. 数据处理与特征筛选
    myfea = AiFeatureMiner(data.copy())
    try:
        myfea.my_feature_select(
            to_drop=[], y_target='label',
            empty_p=empty_p, iv_p=iv_p, corr_p=corr_p, must_var=[]
        )
        selected_data = myfea.selected_dataframe
    except Exception as e:
        print(f"参数组合 empty_p={empty_p:.2f}, iv_p={iv_p:.3f}, corr_p={corr_p:.2f} 导致错误: {e}")
        return 0.0  # 如果特征筛选失败，直接返回一个极差的分数，告诉 Optuna 这个组合不行
        
    if selected_data.shape[1] <= 2: # 如果特征被删光了
        print(f"参数组合 empty_p={empty_p:.2f}, iv_p={iv_p:.3f}, corr_p={corr_p:.2f} 导致特征过少，跳过")
        return 0.0
        
    X = selected_data.drop(['label'], axis=1, errors='ignore')
    y = selected_data['label']
    
    # 3. 快速模型验证
    clf = LGBMClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    score = cross_val_score(clf, X, y, cv=3, scoring='roc_auc').mean()
    
    return score # Optuna 会自动去最大化这个得分

# 开始智能搜索 (只找 30 次就能逼近最优解)
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

print(f"最佳参数组合: {study.best_params}")
print(f"最高 AUC: {study.best_value}")

[I 2026-06-24 16:30:12,099] A new study created in memory with name: no-name-2efdb44a-b796-4cbc-afd9-5c595f786676


数据集加载完成，数据形状: (200, 364)


[I 2026-06-24 16:30:12,585] Trial 0 finished with value: 0.0 and parameters: {'empty_p': 0.98007835500668, 'iv_p': 0.021052487888673067, 'corr_p': 0.7124538026569553}. Best is trial 0 with value: 0.0.


参数组合 empty_p=0.98, iv_p=0.021, corr_p=0.71 导致特征过少，跳过


[I 2026-06-24 16:30:13,050] Trial 1 finished with value: 0.0 and parameters: {'empty_p': 0.9559035782053771, 'iv_p': 0.04160431759734086, 'corr_p': 0.9080119495517021}. Best is trial 0 with value: 0.0.


...

参数组合 empty_p=0.98, iv_p=0.021, corr_p=0.85 导致特征过少，跳过


[I 2026-06-24 16:30:26,000] Trial 28 finished with value: 0.0 and parameters: {'empty_p': 0.9751858542360938, 'iv_p': 0.022005146784543572, 'corr_p': 0.798042108910184}. Best is trial 0 with value: 0.0.


参数组合 empty_p=0.98, iv_p=0.022, corr_p=0.80 导致特征过少，跳过


[I 2026-06-24 16:30:26,461] Trial 29 finished with value: 0.0 and parameters: {'empty_p': 0.9124025815527275, 'iv_p': 0.015533207158595344, 'corr_p': 0.7600608182633666}. Best is trial 0 with value: 0.0.


参数组合 empty_p=0.91, iv_p=0.016, corr_p=0.76 导致特征过少，跳过
最佳参数组合: {'empty_p': 0.98007835500668, 'iv_p': 0.021052487888673067, 'corr_p': 0.7124538026569553}
最高 AUC: 0.0


In [ ]:
#XGBoost
import optuna
import pandas as pd
from sklearn.model_selection import cross_val_score
import xgboost as xgb
import optuna
import pandas as pd
import numpy as np
import pandas as pd
import toad
import matplotlib.pyplot as plt
from aidataprocess import AiDataProcess
from aifeatureminer import AiFeatureMiner
import hashlib


data = pd.read_csv('F:\\datamining\\sensor_data260624\\normalize_feature\\feature\\combined_features.csv')
print(f"数据集加载完成，数据形状: {data.shape}")

def objective_xgb(trial):
    # 1. 智能参数推荐
    empty_p = trial.suggest_float('empty_p', 0.90, 0.99)
    iv_p = trial.suggest_float('iv_p', 0.01, 0.05)
    corr_p = trial.suggest_float('corr_p', 0.70, 0.95)
    
    # 2. 特征筛选
    myfea = AiFeatureMiner(data.copy())
    try:
        myfea.my_feature_select(
            to_drop=[], y_target='label',
            empty_p=empty_p, iv_p=iv_p, corr_p=corr_p, must_var=[]
        )
        selected_data = myfea.selected_dataframe
    except Exception as e:
        print(f"参数组合 empty_p={empty_p:.2f}, iv_p={iv_p:.3f}, corr_p={corr_p:.2f} 导致错误: {e}")
        return 0.0
        
    X = selected_data.drop(['label'], axis=1, errors='ignore')
    y = selected_data['label']
    
    # ===== 修改：特征数目必须介于20到30之间 =====
    if X.shape[1] < 20 or X.shape[1] > 30:
        print(f"参数组合 empty_p={empty_p:.2f}, iv_p={iv_p:.3f}, corr_p={corr_p:.2f} "
              f"特征数={X.shape[1]}，不在[20,30]范围内，跳过")
        return 0.0
    # ===========================================
    
    # 3. XGBoost 模型配置与验证
    clf = xgb.XGBClassifier(
        n_estimators=50, 
        random_state=42, 
        n_jobs=-1, 
        eval_metric='auc'
    )
    
    score = cross_val_score(clf, X, y, cv=3, scoring='roc_auc').mean()
    return score

print("🚀 开始使用 XGBoost 进行特征参数寻优...")
study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=30)

print("\n" + "="*50)
print("🏆 XGBoost 特征筛选参数寻优完成！")
print(f"最佳参数组合: {study_xgb.best_params}")
print(f"最高 AUC 得分: {study_xgb.best_value:.4f}")
print("="*50)

[I 2026-06-24 16:30:30,276] A new study created in memory with name: no-name-9dc947fc-3919-4347-8df4-a4a389a301c5


数据集加载完成，数据形状: (200, 364)
🚀 开始使用 XGBoost 进行特征参数寻优...


[I 2026-06-24 16:30:30,774] Trial 0 finished with value: 0.0 and parameters: {'empty_p': 0.9399624296980081, 'iv_p': 0.024110601397971297, 'corr_p': 0.7624178246495841}. Best is trial 0 with value: 0.0.


参数组合 empty_p=0.94, iv_p=0.024, corr_p=0.76 导致特征过少，跳过


[I 2026-06-24 16:30:31,301] Trial 1 finished with value: 0.0 and parameters: {'empty_p': 0.9264896560393068, 'iv_p': 0.018835394901866694, 'corr_p': 0.7309584063612884}. Best is trial 0 with value: 0.0.


...

参数组合 empty_p=0.93, iv_p=0.042, corr_p=0.88 导致特征过少，跳过


[I 2026-06-24 16:30:47,098] Trial 28 finished with value: 0.0 and parameters: {'empty_p': 0.9621234815200365, 'iv_p': 0.049821720813729986, 'corr_p': 0.8251358541283347}. Best is trial 0 with value: 0.0.


参数组合 empty_p=0.96, iv_p=0.050, corr_p=0.83 导致特征过少，跳过


[I 2026-06-24 16:30:47,623] Trial 29 finished with value: 0.0 and parameters: {'empty_p': 0.9891255639665613, 'iv_p': 0.03313869283626262, 'corr_p': 0.7508907713215942}. Best is trial 0 with value: 0.0.


参数组合 empty_p=0.99, iv_p=0.033, corr_p=0.75 导致特征过少，跳过

🏆 XGBoost 特征筛选参数寻优完成！
最佳参数组合: {'empty_p': 0.9399624296980081, 'iv_p': 0.024110601397971297, 'corr_p': 0.7624178246495841}
最高 AUC 得分: 0.0000


In [1]:
#CatBoost
import optuna
import pandas as pd
from sklearn.model_selection import cross_val_score
import catboost as cb
import optuna
from sklearn.model_selection import cross_val_score
from lightgbm import LGBMClassifier 
import pandas as pd
import numpy as np
import pandas as pd
import toad
import matplotlib.pyplot as plt
from aidataprocess import AiDataProcess
from aifeatureminer import AiFeatureMiner
import hashlib


data = pd.read_csv('F:\\datamining\\sensor_data260624\\normalize_feature\\feature\\combined_features.csv')
print(f"数据集加载完成，数据形状: {data.shape}")

def objective_cat(trial):
    # 1. 智能参数推荐
    empty_p = trial.suggest_float('empty_p', 0.0, 0.0)
    iv_p = trial.suggest_float('iv_p', 0.01, 0.01)
    corr_p = trial.suggest_float('corr_p', 0.0, 0.0)
    
    # 2. 特征筛选
    myfea = AiFeatureMiner(data.copy())
    try:
        myfea.my_feature_select(
            to_drop=[], y_target='label',
            empty_p=empty_p, iv_p=iv_p, corr_p=corr_p, must_var=[]
        )
        selected_data = myfea.selected_dataframe
    except Exception as e:
        print(f"参数组合 empty_p={empty_p:.2f}, iv_p={iv_p:.3f}, corr_p={corr_p:.2f} 导致错误: {e}")
        return 0.0
        
    if selected_data.shape[1] <= 2: 
        print(f"参数组合 empty_p={empty_p:.2f}, iv_p={iv_p:.3f}, corr_p={corr_p:.2f} 导致特征过少，跳过")
        return 0.0
        
    X = selected_data.drop(['label'], axis=1, errors='ignore')
    y = selected_data['label']
    
    # 3. CatBoost 模型配置与验证
    clf = cb.CatBoostClassifier(
        iterations=50,       # 对应 XGB/LGBM 的 n_estimators
        random_state=42, 
        thread_count=-1,     # 对应 n_jobs，调用所有CPU核心
        verbose=0,           # 核心防坑：禁止打印训练过程，否则会严重刷屏
        eval_metric='AUC'
    )
    
    score = cross_val_score(clf, X, y, cv=3, scoring='roc_auc').mean()
    return score

print("🐆 开始使用 CatBoost 进行特征参数寻优...")
study_cat = optuna.create_study(direction="maximize")
study_cat.optimize(objective_cat, n_trials=30)

print("\n" + "="*50)
print("🏆 CatBoost 特征筛选参数寻优完成！")
print(f"最佳参数组合: {study_cat.best_params}")
print(f"最高 AUC 得分: {study_cat.best_value:.4f}")
print("="*50)

f:\Anaconda\envs\datamining\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-06-24 16:26:29,274] A new study created in memory with name: no-name-ce76a8e8-186c-47e7-86f4-376c7880d816


数据集加载完成，数据形状: (200, 364)
🐆 开始使用 CatBoost 进行特征参数寻优...


[I 2026-06-24 16:26:29,839] Trial 0 finished with value: 0.0 and parameters: {'empty_p': 0.0, 'iv_p': 0.01, 'corr_p': 0.0}. Best is trial 0 with value: 0.0.


参数组合 empty_p=0.00, iv_p=0.010, corr_p=0.00 导致特征过少，跳过


[I 2026-06-24 16:26:30,348] Trial 1 finished with value: 0.0 and parameters: {'empty_p': 0.0, 'iv_p': 0.01, 'corr_p': 0.0}. Best is trial 0 with value: 0.0.


...

参数组合 empty_p=0.00, iv_p=0.010, corr_p=0.00 导致特征过少，跳过


[I 2026-06-24 16:26:43,474] Trial 28 finished with value: 0.0 and parameters: {'empty_p': 0.0, 'iv_p': 0.01, 'corr_p': 0.0}. Best is trial 0 with value: 0.0.


参数组合 empty_p=0.00, iv_p=0.010, corr_p=0.00 导致特征过少，跳过


[I 2026-06-24 16:26:43,948] Trial 29 finished with value: 0.0 and parameters: {'empty_p': 0.0, 'iv_p': 0.01, 'corr_p': 0.0}. Best is trial 0 with value: 0.0.


参数组合 empty_p=0.00, iv_p=0.010, corr_p=0.00 导致特征过少，跳过

🏆 CatBoost 特征筛选参数寻优完成！
最佳参数组合: {'empty_p': 0.0, 'iv_p': 0.01, 'corr_p': 0.0}
最高 AUC 得分: 0.0000
